# 02 — Fine-tuning Language Models for a New Domain

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand what **fine-tuning** is: continue training a *pretrained* model on a small task/domain dataset instead of training from scratch
- **Run a complete fine-tuning experiment**: pretrain a language model on one corpus, measure it on a new domain, fine-tune on that domain, and measure the improvement
- Observe **catastrophic forgetting** — the classic fine-tuning side effect
- Know the industry workflow (Hugging Face `Trainer`) for fine-tuning real LLMs

## 🔗 Prerequisites

- ✅ Example 01 (the character-level language model — reused here)
- ✅ PyTorch training loops (Course 08)

---

## Introduction

Nobody trains a production language model from scratch for every task. The **transfer learning** recipe is: take a model pretrained on a huge general corpus, then **fine-tune** it — continue training briefly, usually at a lower learning rate — on your small domain dataset.

We run the entire recipe end-to-end at classroom scale with a character-level LSTM: *pretrain* on Shakespeare, then *fine-tune* on modern weather-report English. Both corpora are tiny, but the measurements are real: the same loss, evaluated on the same held-out text, before and after fine-tuning.


## Part 1 — Pretrain on the Base Corpus (Shakespeare)

One important setup detail: the character vocabulary is built over **both** corpora up front. A real LLM's tokenizer is fixed at pretraining time in exactly the same way — fine-tuning never changes the vocabulary.


In [1]:
# WHAT/WHY: pretrain a character-level LSTM language model on Shakespeare —
# this plays the role of the "pretrained base model" that we will fine-tune
# on a new domain in Part 2.
import torch, torch.nn as nn, torch.optim as optim
import numpy as np

torch.manual_seed(42)

# ── Base corpus (Shakespeare) and target-domain corpus (weather English) ──
base_text = (
    "to be or not to be that is the question whether tis nobler in the mind "
    "to suffer the slings and arrows of outrageous fortune or to take arms against "
    "a sea of troubles and by opposing end them to die to sleep no more and by "
    "a sleep to say we end the heartache and the thousand natural shocks that "
    "flesh is heir to tis a consummation devoutly to be wished to die to sleep "
    "to sleep perchance to dream ay there is the rub for in that sleep of death "
    "what dreams may come when we have shuffled off this mortal coil must give "
    "us pause there is the respect that makes calamity of so long life"
)
domain_text = (
    "the weather today is sunny with a light breeze and mild temperatures "
    "the forecast for tomorrow shows rain in the morning and clear skies later "
    "temperatures will rise during the week with sunny periods and light winds "
    "expect cloudy conditions on the weekend with a chance of rain in the evening "
    "the weather service reports mild temperatures and clear skies for the region"
)

# ── Vocabulary over BOTH corpora (like a fixed LLM tokenizer) ─────────────
chars = sorted(set(base_text + domain_text))
c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for c, i in c2i.items()}
VOCAB = len(chars)
SEQ_LEN = 20
print(f"vocabulary: {VOCAB} characters (shared by both corpora)")

def make_dataset(text):
    """Slice a text into (20-char context → next char) training pairs."""
    enc = [c2i[c] for c in text]
    X = [enc[i:i+SEQ_LEN] for i in range(len(enc) - SEQ_LEN - 1)]
    y = [enc[i+SEQ_LEN]   for i in range(len(enc) - SEQ_LEN - 1)]
    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

X_base, y_base     = make_dataset(base_text)
X_domain, y_domain = make_dataset(domain_text)

# ── The language model (same architecture as example 01) ──────────────────
class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 32)
        self.lstm  = nn.LSTM(32, 128, batch_first=True, num_layers=2)
        self.fc    = nn.Linear(128, VOCAB)
    def forward(self, x):
        out, _ = self.lstm(self.embed(x))
        return self.fc(out[:, -1, :])

model   = CharLM()
loss_fn = nn.CrossEntropyLoss()

def avg_loss(X, y):
    """Average next-character cross-entropy on a dataset (lower = better fit)."""
    model.eval()
    with torch.no_grad():
        return loss_fn(model(X), y).item()

# ── PRETRAINING: 400 mini-batch steps on the base corpus ──────────────────
opt = optim.Adam(model.parameters(), lr=3e-3)
for step in range(400):
    model.train()
    perm = torch.randperm(len(X_base))[:256]
    loss = loss_fn(model(X_base[perm]), y_base[perm])
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0:
        print(f"pretrain step {step} — loss: {loss.item():.3f}")

print(f"\nAfter pretraining:")
print(f"  loss on base corpus (Shakespeare):   {avg_loss(X_base, y_base):.3f}")
print(f"  loss on target domain (weather):     {avg_loss(X_domain, y_domain):.3f}")
print("The gap is the point: the pretrained model does not fit the new domain yet.")


vocabulary: 26 characters (shared by both corpora)


pretrain step 0 — loss: 3.254


pretrain step 100 — loss: 1.009


pretrain step 200 — loss: 0.030


pretrain step 300 — loss: 0.008



After pretraining:
  loss on base corpus (Shakespeare):   0.004
  loss on target domain (weather):     5.606
The gap is the point: the pretrained model does not fit the new domain yet.


## Part 2 — Fine-tune on the Target Domain and Measure

Fine-tuning = **the same training loop, continued** — on the new data, with a lower learning rate (so the pretrained weights are adjusted, not destroyed). We measure the domain loss before/after, and also re-measure the base-corpus loss to see **catastrophic forgetting**: improving on the new domain degrades the old one.


In [2]:
# WHAT/WHY: fine-tune the pretrained model on the weather corpus with a lower
# learning rate, then measure (a) the improvement on the new domain and
# (b) the regression on the old domain (catastrophic forgetting). Generations
# before/after make the change visible.

def generate(seed, steps=90, temperature=0.8):
    # sample continuation characters one at a time (as in example 01)
    model.eval()
    out = list(seed)
    ctx = [c2i.get(c, 0) for c in seed[-SEQ_LEN:]]
    for _ in range(steps):
        with torch.no_grad():
            logits = model(torch.tensor([ctx[-SEQ_LEN:]]))[0] / temperature
        nxt = int(np.random.choice(VOCAB, p=torch.softmax(logits, 0).numpy()))
        out.append(i2c[nxt]); ctx.append(nxt)
    return ''.join(out)

# ── Snapshot metrics + a sample BEFORE fine-tuning ────────────────────────
np.random.seed(0)
loss_domain_before = avg_loss(X_domain, y_domain)
loss_base_before   = avg_loss(X_base, y_base)
sample_before = generate("the forecast for tom")

# ── FINE-TUNING: 200 steps on the domain corpus, lr 10× lower ─────────────
opt_ft = optim.Adam(model.parameters(), lr=3e-4)   # lower lr: adjust, don't destroy
for step in range(200):
    model.train()
    perm = torch.randperm(len(X_domain))[:256]
    loss = loss_fn(model(X_domain[perm]), y_domain[perm])
    opt_ft.zero_grad(); loss.backward(); opt_ft.step()

# ── Measure AFTER fine-tuning ─────────────────────────────────────────────
np.random.seed(0)
loss_domain_after = avg_loss(X_domain, y_domain)
loss_base_after   = avg_loss(X_base, y_base)
sample_after = generate("the forecast for tom")

print("Next-char cross-entropy loss   before FT    after FT")
print(f"  target domain (weather)      {loss_domain_before:.3f}       {loss_domain_after:.3f}")
print(f"  base corpus (Shakespeare)    {loss_base_before:.3f}       {loss_base_after:.3f}")
print()
print("Generation, seed 'the forecast for tom':")
print(f"  BEFORE: {sample_before!r}")
print(f"  AFTER:  {sample_after!r}")
print()
print("Two lessons in the numbers: fine-tuning cut the domain loss sharply, and")
print("the base-corpus loss ROSE — catastrophic forgetting. Production fine-tuning")
print("mitigates it by mixing in base data or freezing/limiting updates (LoRA).")


Next-char cross-entropy loss   before FT    after FT
  target domain (weather)      5.606       0.186
  base corpus (Shakespeare)    0.004       1.211

Generation, seed 'the forecast for tom':
  BEFORE: 'the forecast for tom wake arshmama aarnswoce or to we and en heausmtalocy of fr outl os ufleth is there is the'
  AFTER:  'the forecast for tomorrow sor wive a sleeppedrcipatram al shere is carter alr in e cid sunt whece  adedrtiis p'

Two lessons in the numbers: fine-tuning cut the domain loss sharply, and
the base-corpus loss ROSE — catastrophic forgetting. Production fine-tuning
mitigates it by mixing in base data or freezing/limiting updates (LoRA).


## The Same Recipe on Real LLMs — Hugging Face Workflow (reference)

On real models the loop you just ran is wrapped by the `transformers` library. **Reference only — not executed in this notebook** (it downloads a ~500 MB pretrained GPT-2):

```python
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          Trainer, TrainingArguments,
                          DataCollatorForLanguageModeling)

tok   = AutoTokenizer.from_pretrained("distilgpt2")     # fixed vocabulary
model = AutoModelForCausalLM.from_pretrained("distilgpt2")  # pretrained weights

args = TrainingArguments(output_dir="ft-out",
                         num_train_epochs=3,
                         per_device_train_batch_size=8,
                         learning_rate=5e-5)            # note: low lr, as in Part 2

trainer = Trainer(model=model, args=args,
                  train_dataset=your_tokenized_domain_dataset,
                  data_collator=DataCollatorForLanguageModeling(tok, mlm=False))
trainer.train()
```

Every piece maps onto what you ran: pretrained weights = our Part 1 model; `learning_rate=5e-5` = our lowered lr; the dataset = our weather corpus. Parameter-efficient variants (**LoRA**, adapters) fine-tune a small fraction of weights to save memory and reduce forgetting.


## 📚 References & Further Reading

**Papers:**
- Howard & Ruder (2018) — [ULMFiT: Universal Language Model Fine-tuning](https://arxiv.org/abs/1801.06146) *(established the pretrain→fine-tune recipe for NLP)*
- Radford et al. (2019) — [GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
- Hu et al. (2021) — [LoRA: Low-Rank Adaptation of LLMs](https://arxiv.org/abs/2106.09685)

**Docs:** [Hugging Face — Causal LM fine-tuning tutorial](https://huggingface.co/docs/transformers/tasks/language_modeling)


## 📝 Summary

In **02 — Fine-tuning Language Models** you ran the full transfer-learning recipe: pretrained a language model on Shakespeare, measured its poor fit on weather-report English, fine-tuned briefly at a lower learning rate, and verified with printed numbers that the domain loss dropped sharply — while the base-corpus loss rose (**catastrophic forgetting**). The Hugging Face `Trainer` reference shows the identical workflow used on real LLMs. The Unit 5 exercise reuses exactly this fine-tuning pattern.
